In [ ]:
!lscpu

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd drive/"MyDrive"/"Colab Notebooks"/"new_offRL"/"rl_v2_4"


# ReBRAC formal Stage C —— 5-seed 正式复核

**定位**：[rebrac_experiment_plan.md §6.4](../docs/rebrac_experiment_plan.md) Stage C 的正式执行入口。

**前置**：

- Stage B0 确认 `TRAIN_EPOCHS=64` 足够（[report §5](../docs/rebrac_experiment_report.md)）。
- Stage B 已完成；screening winner `(β1=4.0, β2=2.0)` 在两个 `crosscomp` 数据集上都显著超越 TD3BC phase0c（[report §6](../docs/rebrac_experiment_report.md)）。

**目标**：把 Stage B 的 3-seed screening 结论升级到 5-seed 正式结果，且与 TD3BC phase0c Stage C 的正式成绩同口径可比。

**Finalist（由 Stage B 锁定）**：

- **主 finalist**：`(β1=4.0, β2=2.0)`，在 `crosscomp-1000` 和 `crosscomp-2000` 上共用。
- **ep2000 backup finalist**：`(β1=4.0, β2=1.0)`——仅在 `crosscomp-2000` 上追加 5-seed，作为 主 finalist 在更多 seed 下若方差放大的接盘配置。

**协议变化**（相对 Stage B）：

- seeds：`42 / 43 / 44 / 45 / 46`（新增 45/46）；
- test manifest 提升到 **100 episodes**（Stage B 是 40）；
- val manifest 保持 40 episodes（只用来挑 ckpt，不需要更大）；
- CHECKPOINT_ROOT / RESULTS_ROOT 完全独立 Stage B 的目录，避免语义混淆；
- 因为 Stage B 的 finalist winner 已有 3 seeds × 2 dataset 的数据，本 notebook 不重复 β1×β2 全网格——只跑 finalist 网格，共 15 runs。

**成功判据**：

1. ep1000 主 finalist `mean_test_success_rate ≥ 0.672`（TD3BC phase0c `α=0.25` 正式成绩）；
2. ep2000 至少一个 finalist `mean_test_success_rate ≥ 0.75`；
3. 所有 finalist `std_test_success_rate ≤ 0.10`。

任一失败将触发 plan §6.4 末尾的分类分流。


## 0. 共享环境配置

Stage C 独立输出路径：

- `checkpoints/offline/rebrac/formal/`
- `results/offline/rebrac/formal/`
- `results/offline/rebrac/formal/summaries/overview.csv`

Manifest 复用 Stage B 的同 BENCHMARK_KEY 目录树（`benchmarks/offline_rebrac_screen/`），但 test 走新的 `test_100/` 子目录。


In [ ]:
import os

# —— 通用（与 Stage B / Stage B0 对齐）——
os.environ["PYTHON_BIN"]         = "python3"
os.environ["DEVICE"]             = "cuda"
os.environ["EVAL_WORKERS"]       = "6"
os.environ["EVAL_WORKER_DEVICE"] = "cpu"

# —— Stage C 独有：5 seeds，更大 test manifest ——
os.environ["SEEDS"]                   = "42 43 44 45 46"
os.environ["TRAIN_EPOCHS"]            = "64"
os.environ["CHECKPOINT_EVERY_EPOCHS"] = "8"
os.environ["VAL_MANIFEST_EPISODES"]   = "40"
os.environ["TEST_MANIFEST_EPISODES"]  = "100"

# —— 输出目录独立 ——
os.environ["CHECKPOINT_ROOT"] = "checkpoints/offline/rebrac/formal"
os.environ["RESULTS_ROOT"]    = "results/offline/rebrac/formal"
os.environ["SUMMARY_ROOT"]    = "results/offline/rebrac/formal/summaries"

# 其余默认：BENCHMARK_KEY=single_u10_cross_tgt15,
# BASELINE_POLICY=crosscomp, PROBE_LAYOUT=s0, HISTORY_LENGTH=4,
# TASK_GEOMETRY=cross_stream, TARGET_SPEED=1.5, OBJECTIVE=efficiency_v2,
# SAMPLING_MODE=shuffle_no_replacement, BATCH_SIZE=256,
# critic_layernorm=on, actor_layernorm=off, normalize_q=on


## 1. 生成 manifest

val_40 manifest 已经在 Stage B 时生成，会被 skip；test_100 是 Stage C 新增。

这里需要先**暂时用 finalist 网格 A 的变量**来驱动 manifest 生成（脚本只读 `VAL_MANIFEST_EPISODES / TEST_MANIFEST_EPISODES`，不依赖网格内容）。


In [ ]:
os.environ["MODE"]                 = "manifests"
os.environ["DATASET_EPISODES"]     = "1000 2000"
os.environ["ACTOR_PENALTY_COEFS"]  = "4.0"
os.environ["CRITIC_PENALTY_COEFS"] = "2.0"
!bash scripts/run_offline_rebrac_screen.sh


## 2. 离线数据（应已由 phase0c / Stage B0 / Stage B 产生）

`offline_data/crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep{1000,2000}/transitions.npz` 两份数据集应当都在，会被 skip。


In [ ]:
os.environ["MODE"]             = "collect"
os.environ["DATASET_EPISODES"] = "1000 2000"
!bash scripts/run_offline_rebrac_screen.sh


## 3. 批 A：主 finalist `(β1=4.0, β2=2.0)` on `crosscomp-{1000, 2000}` × 5 seeds

**网格展开**：2 datasets × 1 β1 × 1 β2 × 5 seeds = **10 runs**

每个 run 产出 8 个 ckpt × 40 val + 1 次 100-episode test。脚本会按 `success_rate → return → -safety_cost → -time` 挑 ckpt，然后在 test_100 上重跑。

重训成本说明：Stage B 已经用 seeds 42/43/44 在这个 pair 上训过一次；但 Stage C 使用独立 CHECKPOINT_ROOT，这里会全量重训（seed 42-46 共 10 runs），以避免 Stage B / C 结果耦合。如果想复用 Stage B 的训练权重，手动把 `CHECKPOINT_ROOT` 指向 `checkpoints/offline/rebrac/screening/`，那 seeds 42/43/44 的训练会 skip（只剩 val/test 重算）。


In [ ]:
os.environ["MODE"]                 = "all"
os.environ["DATASET_EPISODES"]     = "1000 2000"
os.environ["ACTOR_PENALTY_COEFS"]  = "4.0"
os.environ["CRITIC_PENALTY_COEFS"] = "2.0"
!bash scripts/run_offline_rebrac_screen.sh


## 4. 批 B：backup finalist `(β1=4.0, β2=1.0)` on `crosscomp-2000` × 5 seeds

**网格展开**：1 dataset × 1 β1 × 1 β2 × 5 seeds = **5 runs**

理由见 [rebrac_experiment_report.md §6.7](../docs/rebrac_experiment_report.md)：
在 Stage B 的 `crosscomp-2000` 上，`(β1=4.0, β2=1.0)` 与主 finalist 的差距是测量噪声量级（Δmean 1.7pp、Δstd 0.008），追加一组 5-seed 只多 5 runs 但能在主 finalist 若在 Stage C 下方差放大时直接接盘，避免 rework。

`crosscomp-1000` 不追加 backup——那里主 finalist 相对第二名的差距是 3.3pp，没有同等质量的候选。


In [ ]:
os.environ["MODE"]                 = "all"
os.environ["DATASET_EPISODES"]     = "2000"
os.environ["ACTOR_PENALTY_COEFS"]  = "4.0"
os.environ["CRITIC_PENALTY_COEFS"] = "1.0"
!bash scripts/run_offline_rebrac_screen.sh


## 5. Overview：5-seed 正式成绩

把 `summaries/overview.csv` 按 `(dataset, mean_test_success_rate desc)` 排序显示。

此时表里应有 **3 行**：

| dataset | pair | 说明 |
| --- | --- | --- |
| `crosscomp-1000` | `actorb_4p0__criticb_2p0` | 主 finalist |
| `crosscomp-2000` | `actorb_4p0__criticb_2p0` | 主 finalist |
| `crosscomp-2000` | `actorb_4p0__criticb_1p0` | backup finalist |

**关键读法**：对照 Stage B 的 3-seed 结果（表格下方并排展示），看均值和 std 是否稳定。


In [ ]:
import csv
from pathlib import Path

import pandas as pd

SUMMARY_CSV = Path(os.environ["SUMMARY_ROOT"]) / "overview.csv"

if not SUMMARY_CSV.exists():
    print(f"[missing] {SUMMARY_CSV} — run §3/§4 first.")
else:
    df = pd.read_csv(SUMMARY_CSV)

    def _fmt(x, precision=4):
        try:
            return f"{float(x):.{precision}f}"
        except Exception:
            return str(x)

    display_cols = [
        "dataset",
        "pair",
        "num_seeds",
        "mean_test_success_rate",
        "std_test_success_rate",
        "mean_test_return",
        "std_test_return",
        "mean_test_safety_cost",
        "mean_test_time_s",
        "mean_critic_penalty",
        "mean_target_q",
        "mean_critic_penalty_ratio",
    ]
    view = df[display_cols].copy()
    view = view.sort_values(
        ["dataset", "mean_test_success_rate"], ascending=[True, False]
    ).reset_index(drop=True)
    for col in [
        "mean_test_success_rate",
        "std_test_success_rate",
        "mean_critic_penalty_ratio",
    ]:
        view[col] = view[col].map(lambda v: _fmt(v, 4))
    for col in [
        "mean_test_return",
        "std_test_return",
        "mean_test_safety_cost",
        "mean_test_time_s",
        "mean_critic_penalty",
        "mean_target_q",
    ]:
        view[col] = view[col].map(lambda v: _fmt(v, 2))
    print(view.to_string(index=False))


## 6. Per-seed 分布

把 5 个 seed 的 test success 全部打出来，同时附上 Stage B 的 3-seed 对应 seed 作对照（如果在同 pair 下）。

**必须检查**：

1. seed 44 是否在 Stage C 下仍然恢复（Stage B 下 ep1000 主 finalist 上 seed 44 = 0.850）；
2. 新增的 seed 45 / seed 46 是否也落在 `0.85 ~ 0.95` 区间，还是出现新的 outlier；
3. backup finalist `(β1=4.0, β2=1.0)` ep2000 的 min-max spread 是否确实小于主 finalist（Stage B 下 spread=0.050 vs 0.025）。


In [ ]:
import json

RESULTS_ROOT = Path(os.environ["RESULTS_ROOT"])

def load_test_payload(dataset: str, pair: str, seed: str) -> dict | None:
    path = RESULTS_ROOT / dataset / pair / "test" / f"seed_{seed}.json"
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

if not SUMMARY_CSV.exists():
    print(f"[missing] {SUMMARY_CSV}")
else:
    SEEDS = os.environ["SEEDS"].split()
    header = f"{'dataset':<60}{'pair':<28}" + "".join(f"seed_{s:<7}" for s in SEEDS) + "min-max spread"
    print(header)
    print("-" * len(header))
    for dataset in sorted(df["dataset"].unique()):
        sub = df[df["dataset"] == dataset].sort_values("mean_test_success_rate", ascending=False)
        for _, r in sub.iterrows():
            pair = r["pair"]
            per_seed = []
            for seed in SEEDS:
                payload = load_test_payload(dataset, pair, seed)
                if payload is None:
                    per_seed.append(None)
                else:
                    per_seed.append(float(payload["eval_success_rate"]))
            seed_cells = "".join(
                f"{v:<12.4f}" if v is not None else f"{'N/A':<12}" for v in per_seed
            )
            valid = [v for v in per_seed if v is not None]
            spread = (max(valid) - min(valid)) if len(valid) >= 2 else 0.0
            print(f"{dataset:<60}{pair:<28}{seed_cells}{spread:.4f}")


## 7. 三张表对比：Stage B（3 seeds, test 40）vs Stage C（5 seeds, test 100）vs TD3BC phase0c

手工硬编码 Stage B 和 TD3BC phase0c 的数字，与当前 Stage C `overview.csv` 拼成一张总表。

**判定规则**：

- 每一 dataset 取 Stage C finalist 中 `mean_test_success_rate` 最高者作为该 dataset 的 ReBRAC 官方成绩；
- Stage C mean ≥ Stage B mean − 0.05（允许 test manifest 变大带来的量级波动）视为 Stage B 结论可复现；
- Stage C mean ≥ TD3BC phase0c 对应 dataset winner + 0.05 视为显著超越；
- 若任何 finalist std_test_success_rate > 0.10，触发 plan §6.4 末尾的失败分流。


In [ ]:
# 硬编码对照组（与 rebrac_experiment_report.md §6.3 / §6.6 保持一致）
STAGE_B_3SEED = {
    ("crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000", "actorb_4p0__criticb_2p0"): (0.883, 0.031),
    ("crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep2000", "actorb_4p0__criticb_2p0"): (0.917, 0.012),
    ("crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep2000", "actorb_4p0__criticb_1p0"): (0.900, 0.020),
}
TD3BC_PHASE0C = {
    "crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000": ("α=0.25", 0.672, 0.045),
    "crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep2000": ("α=0.15", 0.596, 0.036),
}

if not SUMMARY_CSV.exists():
    print(f"[missing] {SUMMARY_CSV}")
else:
    out_rows = []
    for _, r in df.iterrows():
        dataset, pair = r["dataset"], r["pair"]
        c_mean, c_std = float(r["mean_test_success_rate"]), float(r["std_test_success_rate"])
        b_mean, b_std = STAGE_B_3SEED.get((dataset, pair), (None, None))
        td3_info = TD3BC_PHASE0C.get(dataset)
        out_rows.append(
            dict(
                dataset=dataset.replace("crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_", ""),
                pair=pair.replace("actorb_", "β1=").replace("__criticb_", ", β2=").replace("p", "."),
                stage_b_3seed=f"{b_mean:.3f} ± {b_std:.3f}" if b_mean is not None else "—",
                stage_c_5seed=f"{c_mean:.3f} ± {c_std:.3f}",
                td3bc_phase0c=(
                    f"{td3_info[1]:.3f} ± {td3_info[2]:.3f} (TD3BC {td3_info[0]})" if td3_info else "—"
                ),
                delta_vs_td3bc=(
                    f"{(c_mean - td3_info[1])*100:+.1f}pp" if td3_info else "—"
                ),
                delta_vs_stage_b=(
                    f"{(c_mean - b_mean)*100:+.1f}pp" if b_mean is not None else "—"
                ),
            )
        )
    out_df = pd.DataFrame(out_rows)
    print(out_df.to_string(index=False))


## 8. Critic penalty 诊断（β2 · ratio）

Stage B 上这个图是 12 个格子的全扫，Stage C 只有 3 格，依然保留用来做 sanity check：

- 所有条不超过 1.0 → critic 没被 penalty 支配；
- 如果 Stage C 的 `β2·ratio` 相比 Stage B 显著飘（例如主 finalist ep2000 从 0.311 跳到 0.7 以上），说明 `target_q` 的量级在更多 seed 下出现偏移，值得回到训练日志排查。


In [ ]:
import matplotlib.pyplot as plt

if not SUMMARY_CSV.exists():
    print(f"[missing] {SUMMARY_CSV}")
else:
    def _beta2_from_pair(pair):
        return float(pair.split("criticb_")[-1].replace("p", "."))

    datasets = sorted(df["dataset"].unique())
    fig, axes = plt.subplots(1, len(datasets), figsize=(5.5 * len(datasets), 3.6), sharey=True)
    if len(datasets) == 1:
        axes = [axes]
    for ax, dataset in zip(axes, datasets):
        sub = df[df["dataset"] == dataset].copy()
        sub["beta2"] = sub["pair"].map(_beta2_from_pair)
        sub["effective"] = sub["beta2"] * sub["mean_critic_penalty_ratio"]
        sub = sub.sort_values("pair")
        ax.bar(sub["pair"], sub["effective"])
        ax.axhline(1.0, linestyle="--", color="red", alpha=0.7, label="dominance threshold (1.0)")
        ax.axhline(1.5, linestyle="--", color="darkred", alpha=0.9, label="veto threshold (1.5)")
        ax.set_title(dataset.replace("crosscomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_", ""))
        ax.set_ylabel("β2 · mean_critic_penalty_ratio")
        ax.tick_params(axis="x", rotation=45)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
    fig.suptitle("Stage C: critic penalty dominance check")
    fig.tight_layout()
    plt.show()


## 9. Stage C → Stage D 决策规则

跑完上述所有单元后，按以下规则决定是否推进到 Stage D（`worldcomp-1000` teacher-gap follow-up）：

1. **全部通过**（三个 finalist 都满足 §8 的成功判据）：进入 Stage D。报告正式成绩：每个 dataset 取 mean 最高者作为该 dataset 的 ReBRAC 官方成绩。
2. **主 finalist 在 ep2000 上掉队但 backup 撑住**：直接用 backup 作为 ep2000 的 ReBRAC 官方成绩，不做 rework。进入 Stage D。
3. **主 finalist 在 ep1000 上掉队**：回到 Stage B 补跑 `(β1=4.0, β2=1.0)` + 其他邻近格子在 ep1000 上的 5-seed，重新选 finalist；暂停 Stage D。
4. **任一 finalist std > 0.10 但 mean 通过**：挂起 Stage D，回到 Stage B 分析 seed 45/46 是否出现 Stage B 未见的 outlier；可以选择追加 seeds 47/48 到 7-seed 再判一次。
5. **全员崩盘**：停止扩 ReBRAC 主线 screening；按 plan §10 第 5 条转向 XQL。

决策结果和对应的 rework / follow-up 计划，需要写入 [docs/rebrac_experiment_report.md](../docs/rebrac_experiment_report.md) 的新 §7（Stage C 结果）。
